# Build the Neural Network


Neural networks comprise of layers/modules that perform operations on data. The `torch.nn` namespace provides all the building blocks you need to build your own neural network. Every module in PyTorch subclasses the `nn.Module`. A neural network is a module itself that consists of other modules (layers). This nested structure allows for building and managing complex architectures easily.

In the following sections, we’ll build a neural network to classify images in the FashionMNIST dataset.

In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [1]:
from PIL import Image
print(Image.__version__)

12.2.0


## Get Device for Training

We want to be able to train our model on an accelerator such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [3]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device.")

Using cuda device.


## Define the Class

We define our neural network by subclassing `nn.Module`, and initialize the neural network layers in `__init__`. Every `nn.Module` subclass implements the operations on input data in the `forward` method.

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.linear_relu_stack(x)
        return x

We create an instance of `NeuralNetwork`, and move it to the device, and print its structure.

In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model’s `forward`, along with some background operations. Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class, and dim=1 corresponding to the individual values of each output. We get the prediction probabilities by passing it through an instance of the `nn.Softmax` module.

In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2], device='cuda:0')


In [ ]:
# nn.Softmax() : nn.Module, transform a vector to probability distribution
logits = torch.tensor([2.0, 1.0, 0.1])

softmax = nn.Softmax(dim=0)
probs = softmax(logits)

print(probs)
print(probs.sum())

tensor([0.6590, 0.2424, 0.0986])
tensor(1.0000)


In [8]:
# argmax : return the idx of the max value in the tensor
logits = torch.tensor([2.0, 1.0, 0.1])
pred_class = torch.argmax(logits)
print(pred_class, logits[pred_class])

tensor(0) tensor(2.)


## Model Layers

Let’s break down the layers in the FashionMNIST model. To illustrate it, we will take a sample minibatch of 3 images of size 28x28 and see what happens to it as we pass it through the network.

In [9]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


**nn.Flatten**

We initialize the nn.Flatten layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values ( the minibatch dimension (at dim=0) is maintained).

In [10]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


**nn.Linear**

The linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

In [11]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


**nn.ReLU**

Non-linear activations are what create the complex mappings between the model’s inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use nn.ReLU between our linear layers, but there’s other activations to introduce non-linearity in your model.

In [12]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.5155, -0.1460,  0.0657, -0.0899,  0.2273,  0.3311,  0.0080, -0.1686,
          0.3631,  0.0234, -0.3060,  0.0722, -0.4661,  0.6753,  0.0827,  0.1372,
         -0.5559, -0.2041, -0.1768, -0.1266],
        [-0.8146, -0.3537, -0.0232, -0.3489,  0.2116, -0.1034, -0.1798, -0.0734,
          0.2599, -0.3989,  0.1634,  0.1816, -0.2337,  0.3071,  0.0079,  0.0798,
         -0.4529,  0.0516,  0.1079, -0.2214],
        [-0.7397, -0.2564,  0.0823, -0.0917,  0.3080,  0.3703, -0.0650, -0.1416,
          0.4520, -0.4243, -0.3685, -0.1018, -0.6845,  0.6071, -0.0355,  0.5944,
         -0.7300,  0.0491,  0.0460, -0.1454]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0000, 0.0657, 0.0000, 0.2273, 0.3311, 0.0080, 0.0000, 0.3631,
         0.0234, 0.0000, 0.0722, 0.0000, 0.6753, 0.0827, 0.1372, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.2116, 0.0000, 0.0000, 0.0000, 0.2599,
         0.0000, 0.1634, 0.1816, 0.0000, 0.3071, 0.00

**nn.Sequential**

nn.Sequential is an ordered container of modules. The data is passed through all the modules in the same order as defined. You can use sequential containers to put together a quick network like `seq_modules`.

In [13]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10),
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.2441, -0.0006, -0.0024,  0.0476, -0.1985, -0.0954, -0.2074,  0.0428,
          0.0037,  0.1350],
        [ 0.1812, -0.0348, -0.0254,  0.1782, -0.2688, -0.0854, -0.1970,  0.0632,
          0.0529,  0.2826],
        [ 0.2989, -0.1467, -0.0416,  0.1809, -0.1526, -0.0971, -0.1912,  0.0187,
          0.0917,  0.1625]], grad_fn=<AddmmBackward0>)


**nn.Softmax**

The last linear layer of the neural network returns logits - raw values in [-infty, infty] - which are passed to the nn.Softmax module. The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class. `dim` parameter indicates the dimension along which the values must sum to 1.

In [14]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.1269, 0.0994, 0.0992, 0.1043, 0.0815, 0.0904, 0.0808, 0.1038, 0.0998,
         0.1138],
        [0.1166, 0.0939, 0.0948, 0.1162, 0.0743, 0.0893, 0.0798, 0.1036, 0.1025,
         0.1290],
        [0.1315, 0.0842, 0.0936, 0.1169, 0.0837, 0.0885, 0.0806, 0.0994, 0.1069,
         0.1147]], grad_fn=<SoftmaxBackward0>)


## Model Parameters
Many layers inside a neural network are parameterized, i.e. have associated weights and biases that are optimized during training. Subclassing `nn.Module` automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model’s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and a preview of its values.

In [15]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[ 0.0160,  0.0075, -0.0017,  ...,  0.0324,  0.0110, -0.0309],
        [ 0.0057,  0.0296,  0.0007,  ..., -0.0138,  0.0345,  0.0356]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0331, -0.0202], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[-0.0307,  0.0057,  0.0439,  ..., -0.0216, -0.0208, -0.0158],
        [-0.0186, -0.0322,  0.0326,  ..., -0.0195, -0.0203,  0.0062]],
       device='cuda:0', grad_fn=<Slice